# Tutorial: Custom transforms

The instructions for this tutorial can be found in the [documentation](https://metasmith.readthedocs.io/en/latest/tutorials/custom_transforms.html).

In [11]:
from pathlib import Path
from metasmith.python_api import Agent, ContainerRuntime
from metasmith.python_api import DataTypeLibrary, DataInstanceLibrary, TransformInstanceLibrary
from metasmith.python_api import Source, Logistics
from metasmith.python_api import TargetBuilder, Resources, Size, Duration
from metasmith.python_api import ipynbButtonLink

WORKSPACE = Path("../../").resolve() # back twice since we are in example_resources/tutorials
WORKSPACE

PosixPath('/home/tony/workspace/tools/Metasmith/src/metasmith')

In [3]:
agent_home = Source.FromLocal(WORKSPACE/"msm_home")
smith = Agent(
    home = agent_home,
    runtime=ContainerRuntime.DOCKER,
)

smith.Deploy()

2026-01-30_04-56-00  | deploying agent version [0.14.7] to [/home/tony/workspace/tools/Metasmith/src/metasmith/msm_home]
2026-01-30_04-56-00  | [/home/tony/workspace/tools/Metasmith/src/metasmith/msm_home] already exists, use Deploy(assertive=True) to deploy anyways


In [ ]:
local_input_file = WORKSPACE/"epi300.gbk"

mover = Logistics()
mover.QueueTransfer(
    src=Source.FromHttp(url="https://github.com/hallamlab/MetasmithLibraries/releases/download/data.epi300.1/epi300.gbk"),
    dest=Source.FromLocal(local_input_file),
)
mover.ExecuteTransfers()

In [ ]:
MLIB = WORKSPACE/"MetasmithLibraries"
CACHE = WORKSPACE/"cache"
in_dir = CACHE/"inputs/pangenome3.xgdb"

inputs = DataInstanceLibrary(in_dir)
inputs.Purge()
inputs.AddTypeLibrary(MLIB/"data_types/ncbi.yml")
inputs.AddTypeLibrary(MLIB/"data_types/sequences.yml")
inputs.AddTypeLibrary(MLIB/"data_types/pangenome.yml")

group = inputs.AddValue("pangenome", "e coli", "pangenome::pangenome")
inputs.AddValue("DH10b", "GCF_000019425.1", "ncbi::accession", parents={group})
inputs.AddValue("K12", "GCF_000005845.2", "ncbi::accession", parents={group})
inputs.AddItem(WORKSPACE/"epi300.gbk", "sequences::gbk", parents={group})
inputs.Save()

In [ ]:
resources = [
    DataInstanceLibrary.Load(MLIB/f"resources/{n}")
    for n in ["containers", "lib"]
]

transforms = [
    TransformInstanceLibrary.Load(MLIB/f"transforms/{n}")
    for n in ["logistics", "pangenome"]
]

targets = TargetBuilder()
targets.Add("pangenome::heatmap")

task = smith.GenerateWorkflow(
    samples=inputs.AsSamples("ncbi::accession"),
    resources=resources,
    transforms=transforms,
    targets=targets,
)

In [ ]:
print(f'this workflow is called [{len(task.GetKey())}]')

In [ ]:
print(f'generated plan has [{len(task.plan.steps)}] steps')

workflow_diagram_path = task.plan.RenderDAG(WORKSPACE/f"{task.GetKey()}.dag.svg")
ipynbButtonLink(workflow_diagram_path.relative_to(WORKSPACE), "view workflow diagram")

In [ ]:
smith.StageWorkflow(task, on_exist="clear")

In [14]:
preset_path = smith.GetNxfConfigPresets()["slurm"]
# (WORKSPACE/preset_path.name).symlink_to(preset_path)
ipynbButtonLink(f'/files/{preset_path.name}', 'view config')


In [10]:
WORKSPACE

PosixPath('/home/tony/workspace/tools/Metasmith/src/metasmith')

In [ ]:
smith.RunWorkflow(
    task,
    config_file=smith.GetNxfConfigPresets()["local"],
    resource_overrides={
        "all": Resources(
            memory=Size.GB(2),
        )
    }
)

In [ ]:
smith.CheckWorkflow(task)

In [ ]:
results_path = smith.GetResultSource(task).GetPath()
results = DataInstanceLibrary.Load(results_path)

In [ ]:
to_show = [
    "_metadata/logs.latest/nxf_report.html",
    "_metadata/logs.latest/nxf_timeline.html",
] + [path for path, type_name, endpoint in results.Iterate()]

for file in to_show:
    path = results_path.relative_to(WORKSPACE)/file
    ipynbButtonLink(f'/files/{path}', f'view {path.parent.name}/{path.name}')